In [0]:
# 1. Get the list of unique years from your SQL table as a list of strings
year_data = spark.sql("SELECT DISTINCT YEAR(order_in_date) as yr FROM my_project_db.gold_sales_fact ORDER BY yr DESC").collect()
years = [str(row['yr']) for row in year_data]

# 2. Create the dropdown using the dynamic list
dbutils.widgets.dropdown("selected_year", years[0], years)

# 3. Retrieve the selected value for use in subsequent SQL cells
selected_year = dbutils.widgets.get("selected_year")

In [0]:
# 1. Define your fixed values
aging_options = ["30", "60", "90"]

# 2. Create the widget (default to 30)
dbutils.widgets.dropdown("aging_days", "30", aging_options)

# 3. Get the value for use in SQL
aging_val = dbutils.widgets.get("aging_days")

In [0]:
%sql
SELECT 
    p.product_sku,
    p.division,
    YEAR(f.warranty_date) - 2 AS import_year,
    f.warranty_date,
    SUM(f.inventory_qty - f.order_qty) AS total_unsold_qty,
    ROUND(SUM((f.inventory_qty - f.order_qty) * f.purchase_cost), 2) AS total_aging_value
FROM my_project_db.gold_sales_fact f
JOIN my_project_db.gold_dim_products p ON f.invt_id = p.invt_id
WHERE f.warranty_date < date_sub(CURRENT_DATE(), :aging_days)
  AND (f.inventory_qty - f.order_qty) > 0
  -- Added this filter to sync with your dashboard's global slicer
  AND YEAR(f.warranty_date) = :selected_year
GROUP BY 
    p.product_sku, 
    p.division, 
    f.warranty_date
QUALIFY ROW_NUMBER() OVER(PARTITION BY p.product_sku ORDER BY f.warranty_date DESC) = 1
ORDER BY total_aging_value DESC
LIMIT 5;